# Modelado y cierre de Fase 3

## 1. Metodología y evidencia

Se conservan la división temporal del proyecto, las etiquetas de frontera eliminadas y el criterio de filas comunes. El candidato fue fijado por su MAE de validación. Las búsquedas originales permanecen en los notebooks por familia y sus ramas, identificadas en el informe de comparación. Este notebook solo consulta CSV, JSON e imágenes archivados: no carga el dataset procesado, no ajusta modelos y no ejecuta test. Repetir sus celdas no consume otra evaluación.

In [ ]:
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

RAIZ = Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
from auxiliar_functions import tabla_simple

SALIDA = RAIZ / "results/lasso_final"
def leer(nombre):
    return json.loads((SALIDA / nombre).read_text(encoding="utf-8"))
def mostrar(numero, titulo, tabla, precision=6):
    print(f"Tabla {numero}. {titulo}")
    display(tabla_simple(tabla, precision=precision))
meta = leer("entrenamiento.json")
resultado = leer("evaluacion_test.json")
log = pd.read_csv(RAIZ / "results/experiment_logs.csv")
validacion = log.loc[log.tramo.eq("validacion")].sort_values("mae", kind="stable")
mostrar(1, "Registro de validación: todas las ejecuciones históricas", validacion[
    ["modelo", "fecha_hora", "filas", "mae", "rmse", "r2", "hiperparametros", "notas"]])

## 2. Selección fijada por validación

Se presenta la mejor configuración por familia, incluidos los baselines. El registro mantiene por separado las búsquedas iniciales y Optuna aunque repitan resultados. Lasso usa la lista histórica de catorce entradas; no incorpora el indicador agregado posteriormente al contrato de quince. Los resultados de selección de variables de Fase 2 se documentan como antecedentes en comparacion_modelos_final.md y no se reabre la selección de Fase 3.

In [ ]:
mostrar(2, "Mejor configuración por familia en validación", validacion.drop_duplicates("modelo")[
    ["modelo", "filas", "mae", "rmse", "r2", "hiperparametros"]])
mostrar(3, "Configuración ganadora y evidencia original", pd.DataFrame([{
    "Modelo": meta["modelo"], "Pipeline": meta["pipeline"], **meta["hiperparametros"],
    **{f"Validación original {k}": v for k,v in meta["metricas_validacion_original"].items()}}]))
mostrar(4, "Entradas ganadoras en su orden", pd.DataFrame({"Variable": meta["columnas"]}))

## 3. Reentrenamiento y congelamiento

El script train_model.py reconstruye pipeline A y reajusta mediana, escalado estándar y Lasso con entrenamiento más validación. Verifica dos ajustes deterministas y la carga joblib antes de test. El commit del código entrenado se conserva en el registry y el artefacto; el tag identifica el commit posterior que contiene los resultados. Las métricas de ajuste son dentro de muestra y no sustituyen la validación histórica.

In [ ]:
mostrar(5, "Reentrenamiento final y verificaciones previas a test", pd.DataFrame([{
    "Versión": meta["version"], "Train": meta["filas_por_tramo"]["entrenamiento"],
    "Validation": meta["filas_por_tramo"]["validacion"], "Total": meta["filas_train_validation"],
    "Commit de entrenamiento": meta["commit"], "Python": meta["python"], **meta["reproducibilidad"]}]))
mostrar(6, "Métricas de ajuste dentro de muestra, no validación", pd.DataFrame([meta["metricas_ajuste_no_validacion"]]))

## 4. Evaluación única sobre test local

La ejecución explícita crea un marcador exclusivo antes de leer las observaciones de prueba. Modelo y baselines comparten el soporte con objetivo y ambas referencias disponibles. La media móvil sigue la definición del proyecto: siete bloques hasta D, con mínimo de cuatro valores válidos. Las consultas posteriores verifican los archivos guardados y no predicen nuevamente. El umbral eliminatorio de la consigna corresponde al test oculto, no al local.

In [ ]:
comparacion = pd.DataFrame(resultado["metricas"]).T.rename_axis("Modelo")
mostrar(7, "Test local: modelo y baselines sobre las mismas observaciones", comparacion)
mostrar(8, "Soporte de la evaluación única", pd.DataFrame([{
    "Test etiquetado": resultado["filas_test_etiquetadas"], "Filas evaluadas": resultado["filas_evaluadas"],
    "Índices excluidos": str(resultado["indices_excluidos"]), "Predicciones de test": resultado["evaluaciones_predict_test"]}]))
mostrar(9, "Diferencia de MAE y mejora porcentual frente a baselines", pd.DataFrame(resultado["comparaciones"]).T)
display(Image(filename=str(SALIDA / "prediccion_vs_real.png")))

## 5. Residuos, sesgos y errores grandes

Se define residuo como real menos predicho. Un valor positivo indica subestimación; uno negativo, sobreestimación. Se muestran distribución, dispersión y evolución temporal, junto con los mayores errores y el sesgo por cuartil del consumo real. Este diagnóstico es descriptivo: no habilita descartar observaciones ni ajustar el modelo después de test.

In [ ]:
mostrar(10, "Resumen de residuos de test local", pd.DataFrame([leer("resumen_residuos.json")]))
mostrar(11, "Residuos por nivel de consumo real", pd.read_csv(SALIDA / "residuos_por_cuartil.csv"))
mostrar(12, "Observaciones con mayor error absoluto", pd.read_csv(SALIDA / "errores_mayores.csv"))
display(Image(filename=str(SALIDA / "diagnostico_residuos.png")))
display(Image(filename=str(SALIDA / "predicciones_y_residuos_temporales.png")))

## 6. Coeficientes e interpretación

Los coeficientes están expresados en unidades de frío por una desviación estándar del predictor imputado. El signo indica la dirección de la relación condicional. La magnitud permite una comparación descriptiva en la escala estandarizada, pero la correlación entre entradas y las interacciones redistribuyen los coeficientes. No demuestra causalidad ni una importancia estable fuera de muestra.

In [ ]:
mostrar(13, "Coeficientes de Lasso definitivo ordenados por magnitud", pd.read_csv(SALIDA / "coeficientes.csv"))
display(Image(filename=str(SALIDA / "coeficientes.png")))

## 7. Controles de trazabilidad

Las comprobaciones enlazan registry, métricas, eventos y predicciones archivadas. No se abre el dataset ni se usa el estimador. La prueba de reproducibilidad del ajuste se realizó antes de la evaluación única y se conserva como evidencia. Los fallos históricos de auditoría y sus correcciones se explican en la documentación del cierre.

In [ ]:
import hashlib
registro = json.loads((RAIZ / "models/model_registry.json").read_text(encoding="utf-8"))[meta["version"]]
predicciones = pd.read_csv(SALIDA / "predicciones_test.csv")
eventos = log.loc[log.notas.str.contains("evento=evaluacion_final_unica", na=False)]
controles = pd.Series({
    "Catorce entradas sin objetivo ni indicador adicional": len(meta["columnas"]) == 14 and "objetivo" not in meta["columnas"] and "es_fin_de_anio" not in meta["columnas"],
    "Registry conserva la validación original": registro["metricas_validacion_original"] == meta["metricas_validacion_original"],
    "Registry coincide con test": registro["metricas_test_local"] == resultado["metricas"],
    "Modelo conserva su huella": hashlib.sha256((RAIZ / meta["path"]).read_bytes()).hexdigest() == meta["modelo_sha256"],
    "Predicciones conservan su huella": hashlib.sha256((SALIDA / "predicciones_test.csv").read_bytes()).hexdigest() == resultado["predicciones_sha256"],
    "Todos los modelos comparten soporte": len(eventos) == 3 and eventos.filas.eq(len(predicciones)).all(),
    "Residuo coincide con real menos predicho": np.allclose(predicciones.residuo, predicciones.real-predicciones.predicho),
    "Identidades únicas de las filas": predicciones.indice_csv.is_unique,
    "Evaluación única archivada": resultado["evaluaciones_predict_test"] == 1 and leer("test_consumido.json")["estado"] == "completo_no_repetir",
    **meta["reproducibilidad"],
})
mostrar(14, "Controles del cierre", controles.to_frame("Cumple"), precision=0)
assert controles.all(), controles[~controles]

## 8. Conclusión

La validación ya intervino en la selección de variables, por lo que puede ser optimista. La etiqueta representa el bloque siguiente y no está certificada su correspondencia con fecha física ni la disponibilidad operativa a las 23:59. Se separaron índices de ajuste y prueba y se conservó la preparación histórica. La conclusión local se obtiene de los resultados sin revisar la elección del candidato. Fase 4 y test oculto quedan fuera de esta tarea.

In [ ]:
mejor_baseline = min(["persistencia", "media_movil"], key=lambda b: resultado["metricas"][b]["mae"])
mejora = resultado["comparaciones"][mejor_baseline]["mejora_porcentual"]
print(f"Lasso v{meta['version']} obtuvo MAE local {resultado['metricas']['lasso']['mae']:.6f}, frente al MAE de validación histórico {meta['metricas_validacion_original']['mae']:.6f}.")
print(f"La mejora frente al mejor baseline ({mejor_baseline}) fue {mejora:.6f} % sobre {resultado['filas_evaluadas']} observaciones.")
print(f"Supera ambos baselines: {resultado['supera_ambos_baselines']}. MAE local inferior a 3000: {resultado['mae_local_menor_3000']}.")
if mejora < 10:
    print("La ganancia es menor al 10 %. No justificaría desplegar el pipeline por precisión únicamente; aporta interpretación lineal pero requiere mantener imputación, escalado y dependencias. Se conserva el candidato fijado y no se ajusta después de test.")
else:
    print("La mejora local supera el 10 % y aporta evidencia a favor del costo del pipeline, sin garantizar rendimiento futuro ni en test oculto.")
print("La evaluación local no acredita el umbral del test oculto. Los resultados completos de pruebas y trazabilidad están en verificacion_final.json.")